In [163]:
import pandas as pd
import numpy as np
import yfinance as yf
from dotenv import load_dotenv 
import os
import requests


In [164]:
"""
Notes: Something for risk management could be the Kelly Criterion for each bet.
Since we are getting a probalbitity from the market and having our own model.

"""

'\nNotes: Something for risk management could be the Kelly Criterion for each bet.\nSince we are getting a probalbitity from the market and having our own model.\n\n'

Loading Coinbase API Key

In [165]:
load_dotenv()  # take environment variables from .env.

API_KEY = os.getenv("COINBASE_API_KEY")
API_SECRET = os.getenv("COINBASE_API_SECRET")

Pulling Market Data

In [166]:
import requests
import pandas as pd

COINBASE_CANDLES_URL = "https://api.exchange.coinbase.com/products/BTC-USD/candles"
url = COINBASE_CANDLES_URL
params = {"granularity": 900}

r = requests.get(url, params=params)
r.raise_for_status()

df = pd.DataFrame(
    r.json(),
    columns=["timestamp", "low", "high", "open", "close", "volume"]
)

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
df = df.sort_values("timestamp").set_index("timestamp")

print(df.tail())


                          low      high      open     close      volume
timestamp                                                              
2026-02-08 01:15:00  69070.33  69327.36  69327.36  69103.56   30.803676
2026-02-08 01:30:00  69011.31  69315.99  69103.56  69285.98   71.390446
2026-02-08 01:45:00  69210.00  69461.99  69285.97  69327.48   65.578245
2026-02-08 02:00:00  69032.48  69448.61  69333.98  69371.70  160.795901
2026-02-08 02:15:00  69315.13  69657.46  69364.09  69437.98   58.020392


In [167]:
COINBASE_TICKER_URL = "https://api.exchange.coinbase.com/products/BTC-USD/ticker"
r = requests.get(COINBASE_TICKER_URL)
r.raise_for_status()
print("live price:", r.json()["price"])


live price: 69480.68


Developing Functions

In [168]:
def fetch_15m_candles(limit=300):
    # Coinbase candles endpoint returns newest-first unless you specify start/end;
    # for v1 we just fetch recent and sort.
    params = {"granularity": 900}
    r = requests.get(COINBASE_CANDLES_URL, params=params, timeout=10)
    r.raise_for_status()
    df = pd.DataFrame(r.json(), columns=["timestamp","low","high","open","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df = df.sort_values("timestamp").set_index("timestamp").tail(limit)
    return df

In [169]:
def fetch_live_price():
    r = requests.get(COINBASE_TICKER_URL, timeout=10)
    r.raise_for_status()
    return float(r.json()["price"])

In [170]:
from datetime import datetime, timezone

def current_15m_window(ts=None):
    """Return (start, end) of the current 15m window in UTC."""
    if ts is None:
        ts = datetime.now(timezone.utc) #Coordinated Universal Time
    # floor to 15 minutes
    minute = (ts.minute // 15) * 15
    start = ts.replace(minute=minute, second=0, microsecond=0)
    end = start + pd.Timedelta(minutes=15)
    return pd.Timestamp(start), pd.Timestamp(end)

In [171]:
def current_15m_window(ts=None):
    """Return (start, end) of the current 15m window in UTC."""
    if ts is None:
        ts = datetime.now(timezone.utc)
    # floor to 15 minutes
    minute = (ts.minute // 15) * 15
    start = ts.replace(minute=minute, second=0, microsecond=0)
    end = start + pd.Timedelta(minutes=15)
    return pd.Timestamp(start), pd.Timestamp(end)

In [172]:
def compute_features(df_15m, live_price):
    """
    Minimal v1 features inspired by the first article:
    - short-term trend / momentum
    - realized vol proxy
    - stalling after big move
    """
    closes = df_15m["close"].astype(float)

    # Use last 4 candles (1 hour) for basic trend
    last = closes.tail(8)  # 2 hours
    rets = np.log(last).diff().dropna()

    mom_60m = float(np.log(last.iloc[-1] / last.iloc[-5])) if len(last) >= 5 else 0.0
    vol_2h  = float(rets.std()) if len(rets) > 2 else 0.0

    # "stalling": big move then low recent movement
    move_60m = abs(mom_60m)
    recent_move_30m = float(abs(np.log(last.iloc[-1] / last.iloc[-3]))) if len(last) >= 3 else 0.0
    stalling = 1.0 if (move_60m > 0.002 and recent_move_30m < 0.0008) else 0.0  # tweak later

    # live vs last close (micro drift)
    live_vs_lastclose = float(np.log(live_price / closes.iloc[-1]))

    return {
        "mom_60m": mom_60m,
        "vol_2h": vol_2h,
        "stalling": stalling,
        "live_vs_lastclose": live_vs_lastclose,
    }

In [173]:
def model_probability(features):
    """
    Placeholder probability model.
    Replace with a real calibrated model later (logit, XGBoost, etc.)
    For now: momentum sign + stalling adds mean-reversion bias.
    """
    score = 0.0
    score += 2.0 * features["mom_60m"] #60 minute momentum
    score += -1.5 * features["stalling"] * np.sign(features["mom_60m"] or 1)  #trend decay
    score += 1.0 * features["live_vs_lastclose"] #Is the price drifting up or down since the last 15-min interval?

    # squash to 0..1
    p_up = 1.0 / (1.0 + np.exp(-score*50))  # scale factor for sensitivity
    return float(p_up)

In [174]:
def should_trade(p_model, p_mkt, edge_buffer=0.03):
    return (p_model - p_mkt) > edge_buffer

In [175]:
df_15m = fetch_15m_candles()
live = fetch_live_price()
w0, w1 = current_15m_window()

features = compute_features(df_15m, live)
p_up = model_probability(features)

print("Window:", w0, "to", w1)
print("Live:", live)
print("Features:", features)
print("Model P(UP):", round(p_up, 4))

Window: 2026-02-08 02:30:00+00:00 to 2026-02-08 02:45:00+00:00
Live: 69480.36
Features: {'mom_60m': 0.004827731083783765, 'vol_2h': 0.0021917418462447636, 'stalling': 0.0, 'live_vs_lastclose': 0.0006101426394059886}
Model P(UP): 0.6256


Use the Coinbase 15-minute market so we don't have a price difference

In [176]:
"""
Bring in the Kalish Metrics and then double check pricing (do we want to make this an ML model 
or just mean reversion)
"""

'\nBring in the Kalish Metrics and then double check pricing (do we want to make this an ML model \nor just mean reversion)\n'